# Week 1 — EDA & Preprocessing
KuaiRec short video recommendation dataset.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
DATA_DIR = Path('../datasets/KuaiRec 2.0/data')

## 1. Load Data

In [ ]:
big   = pd.read_csv(DATA_DIR / 'big_matrix.csv')
small = pd.read_csv(DATA_DIR / 'small_matrix.csv')
print(f'big_matrix:   {big.shape}   users={big.user_id.nunique():,}  videos={big.video_id.nunique():,}')
print(f'small_matrix: {small.shape}  users={small.user_id.nunique():,}  videos={small.video_id.nunique():,}')
big.head(3)

In [ ]:
# Binary label
big['label']   = (big['watch_ratio']   > 2.0).astype(int)
small['label'] = (small['watch_ratio'] > 2.0).astype(int)
print(f'big_matrix   positive rate: {big.label.mean():.3f}')
print(f'small_matrix positive rate: {small.label.mean():.3f}')

## 2. Watch Ratio Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Clip at 5 for readability
wr_clip = big['watch_ratio'].clip(0, 5)
axes[0].hist(wr_clip, bins=100, color='steelblue', edgecolor='none')
axes[0].axvline(2.0, color='red', linestyle='--', label='threshold=2.0')
axes[0].set_xlabel('watch_ratio (clipped at 5)')
axes[0].set_ylabel('count')
axes[0].set_title('big_matrix watch_ratio distribution')
axes[0].legend()

wr_small = small['watch_ratio'].clip(0, 5)
axes[1].hist(wr_small, bins=100, color='darkorange', edgecolor='none')
axes[1].axvline(2.0, color='red', linestyle='--', label='threshold=2.0')
axes[1].set_xlabel('watch_ratio (clipped at 5)')
axes[1].set_title('small_matrix watch_ratio distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('../experiments/eda_watch_ratio.png', bbox_inches='tight')
plt.show()

print('\nbig_matrix watch_ratio stats:')
print(big['watch_ratio'].describe())

## 3. User Activity Distribution

In [ ]:
user_counts = big.groupby('user_id').size()
print(f'Interactions per user — mean: {user_counts.mean():.0f}  median: {user_counts.median():.0f}  min: {user_counts.min()}  max: {user_counts.max()}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(user_counts, bins=60, color='teal', edgecolor='none')
ax.set_xlabel('# interactions per user')
ax.set_ylabel('# users')
ax.set_title('User activity distribution (big_matrix)')
plt.tight_layout()
plt.savefig('../experiments/eda_user_activity.png', bbox_inches='tight')
plt.show()

## 4. Video Popularity (Long Tail)

In [ ]:
item_counts = big.groupby('video_id').size().sort_values(ascending=False)
print(f'Interactions per video — mean: {item_counts.mean():.0f}  median: {item_counts.median():.0f}  min: {item_counts.min()}  max: {item_counts.max()}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(item_counts)+1), item_counts.values, linewidth=0.8)
ax.set_xlabel('item rank')
ax.set_ylabel('# interactions')
ax.set_title('Item popularity (long-tail) — big_matrix')
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../experiments/eda_item_popularity.png', bbox_inches='tight')
plt.show()

## 5. Temporal Coverage

In [ ]:
big['date_dt'] = pd.to_datetime(big['date'], format='%Y%m%d')
daily = big.groupby('date_dt').size()

fig, ax = plt.subplots(figsize=(12, 4))
daily.plot(ax=ax, color='steelblue')
ax.set_xlabel('date')
ax.set_ylabel('# interactions')
ax.set_title('Daily interaction volume (big_matrix)')
plt.tight_layout()
plt.savefig('../experiments/eda_temporal.png', bbox_inches='tight')
plt.show()

print(f'Date range: {daily.index.min()} → {daily.index.max()}')

## 6. Chronological Train / Val / Test Split

In [ ]:
from src.data.preprocessing import split_big_matrix, load_big_matrix

big_full = load_big_matrix()
train, val, test = split_big_matrix(big_full)

print(f'train → {train.timestamp.min():.0f} to {train.timestamp.max():.0f}')
print(f'val   → {val.timestamp.min():.0f} to {val.timestamp.max():.0f}')
print(f'test  → {test.timestamp.min():.0f} to {test.timestamp.max():.0f}')

# Overlap check
print(f'\ntrain users: {train.user_id.nunique():,}   train items: {train.video_id.nunique():,}')
print(f'val   users: {val.user_id.nunique():,}     val   items: {val.video_id.nunique():,}')
print(f'test  users: {test.user_id.nunique():,}    test  items: {test.video_id.nunique():,}')

## 7. Sequence Statistics

In [ ]:
from src.data.preprocessing import build_watch_sequences

sequences = build_watch_sequences(train, threshold=0.5)
seq_lengths = [len(s) for s in sequences.values()]

print(f'Sequence length — mean: {np.mean(seq_lengths):.0f}  median: {np.median(seq_lengths):.0f}  max: {max(seq_lengths)}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lengths, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel('sequence length')
ax.set_ylabel('# users')
ax.set_title('User watch sequence lengths (threshold=0.5)')
plt.tight_layout()
plt.savefig('../experiments/eda_sequence_lengths.png', bbox_inches='tight')
plt.show()

## 8. Category Distribution

In [ ]:
from src.data.preprocessing import load_caption_category

cat_df = load_caption_category()
top_cats = cat_df['first_level_category_name'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 5))
top_cats.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('# videos')
ax.set_title('Top 20 first-level categories')
plt.tight_layout()
plt.savefig('../experiments/eda_categories.png', bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
summary = {
    'big_matrix rows': len(big),
    'big_matrix users': big.user_id.nunique(),
    'big_matrix videos': big.video_id.nunique(),
    'small_matrix rows': len(small),
    'small_matrix users': small.user_id.nunique(),
    'small_matrix videos': small.video_id.nunique(),
    'positive rate (big, thr=2.0)': round(big.label.mean(), 4),
    'positive rate (small, thr=2.0)': round(small.label.mean(), 4),
    'train size': len(train),
    'val size': len(val),
    'test size': len(test),
    'mean seq len (thr=0.5)': round(np.mean(seq_lengths), 1),
}
for k, v in summary.items():
    print(f'{k:<45} {v:>12,}' if isinstance(v, int) else f'{k:<45} {v}')